# BCCC-DeFiFraudTrans-2025 — Preprocessing & Imbalance Construction

**Notebook 2 of 4.** Produces the clean, split, de-leaked data that notebooks 3 (generation) and 4 (classification) consume.

Decisions baked in (from data exploration):
- **Drop leakage:** `flag` (perfect copy of label), `error_rate`, `number_of_errors` (near-leak / generation artifact)
- **Drop dead/constant:** `status`, `message`, `is_contract_creation`
- **Drop identifiers/junk:** `Unnamed: 0`, `address`, `hash`, `Error!`
- **Drop token-string blocks:** all `erc_20_*`, `erc_721_*`
- **Keep:** core gas / value / nonce / log / address-length behavioral features
- **Split:** real fraud held out for test; training fraud down-sampled to **1:100, 1:500, 1:1000**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/SDG_paper/data/BCC'            # the two CSVs
OUT_DIR  = '/content/drive/MyDrive/SDG_paper/data/processed'      # outputs go here

import os
os.makedirs(OUT_DIR, exist_ok=True)

Mounted at /content/drive


In [ ]:
import pandas as pd, numpy as np, glob, json

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# --- Load both files, label from filename ---
fraud_path = os.path.join(DATA_DIR, 'DeFiTransLyzer_fraud.csv')
legit_path = os.path.join(DATA_DIR, 'DeFiTransLyzer_legitimate.csv')

df_fraud = pd.read_csv(fraud_path, low_memory=False); df_fraud['label'] = 1
df_legit = pd.read_csv(legit_path, low_memory=False); df_legit['label'] = 0
df = pd.concat([df_fraud, df_legit], ignore_index=True)
del df_fraud, df_legit

print('Combined:', df.shape)
print(df['label'].value_counts().to_dict())

Combined: (1026867, 172)
{0: 938074, 1: 88793}


## 1. Drop leakage, dead, identifier, and token-string columns

In [ ]:



leak_cols     = ['flag', 'error_rate', 'number_of_errors']
dead_cols     = ['status', 'message', 'is_contract_creation']
id_junk_cols  = ['Unnamed: 0', 'address', 'hash', 'Error!']
time_cols     = ['block_number', 'chain_id']   # <-- ADD: time proxy + near-constant
token_cols = [c for c in df.columns if c.startswith('erc_20_') or c.startswith('erc_721_')]

to_drop = [c for c in (leak_cols + dead_cols + id_junk_cols + time_cols + token_cols) if c in df.columns]
print(f'Dropping {len(to_drop)} columns ({len(token_cols)} are token-string features).')


df = df.drop(columns=to_drop)
print('Remaining shape:', df.shape)
print('Remaining columns:', [c for c in df.columns if c != 'label'])

Dropping 90 columns (78 are token-string features).
Remaining shape: (1026867, 82)
Remaining columns: ['length_transaction_hash', 'length_to', 'log_removed', 'gas_used', 'length_from', 'index', 'gas_efficiency', 'value', 'total_gas_cost', 'gas_per_log_event', 'log_index', 'event_activity_flag', 'normalized_token_transfer', 'effective_gas_price', 'cumulative_gas_used', 'is_same_address', 'gas_price_ratio', 'token_transfer_amount', 'length_log', 'log_count', 'num_transaction', 'duration_seconds', 'gas_used.gas_used_summation', 'gas_used.gas_used_average', 'gas_used.gas_used_median', 'gas_used.gas_used_standard_deviation', 'gas_used.gas_used_maximum_val', 'gas_used.gas_used_minimum_val', 'gas_used.gas_used_variance', 'gas_used.gas_used_range_value', 'gas_used.gas_used_skewness', 'gas_used.gas_used_mode', 'gas_used.gas_used_coefficient_of_variation', 'gas_prices.gas_prices_summation', 'gas_prices.gas_prices_average', 'gas_prices.gas_prices_median', 'gas_prices.gas_prices_standard_deviation

## 2. Clean remaining columns

Handle the mixed-type columns the DtypeWarning flagged, coerce everything to numeric, and deal with missing values. We keep only numeric behavioral features.

In [ ]:
# token_transfer_amount etc. came in as object (mixed) — coerce to numeric
for col in df.columns:
    if col == 'label':
        continue
    if df[col].dtype == 'object':
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Replace infinities (from ratio features with zero denominators) with NaN
n_inf = np.isinf(df.drop(columns=['label']).select_dtypes(include=[np.number])).sum().sum()
print(f'Infinity values found and neutralized: {int(n_inf)}')
df = df.replace([np.inf, -np.inf], np.nan)

# Drop any column that is entirely NaN after coercion, or constant (zero variance)
nunique = df.drop(columns=['label']).nunique()
constant_cols = nunique[nunique <= 1].index.tolist()
if constant_cols:
    print('Dropping constant columns:', constant_cols)
    df = df.drop(columns=constant_cols)

# Missing values: report, then fill with 0
miss = df.isna().sum()
print('\nColumns with missing values:')
print(miss[miss > 0])
df = df.fillna(0)

print('\nShape after cleaning:', df.shape)

Infinity values found and neutralized: 8252
Dropping constant columns: ['length_transaction_hash', 'log_removed', 'length_from', 'gas_per_log_event', 'log_index', 'normalized_token_transfer', 'log_count']

Columns with missing values:
token_transfer_amount                   8406
num_transaction                           53
duration_seconds                          53
gas_used.gas_used_summation               53
gas_used.gas_used_average                 53
                                        ... 
nonce.nonce_coefficient_of_variation      53
number_of_from_address                    53
number_of_unique_from_address             53
number_of_to_address                      53
number_of_unique_to_address               53
Length: 62, dtype: int64

Shape after cleaning: (1026867, 75)


## 3. Re-check for leakage on the cleaned set

Safety net: flag any remaining feature that separates the classes almost perfectly (a value that is overwhelmingly one class). This catches leaks we didn't anticipate before we train anything.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
import numpy as np

feat_cols = [c for c in df.columns if c != 'label']

# Replace any infinite values or extreme values in df with NaN, then fill with 0 before assigning X_all
df = df.replace([np.inf, -np.inf], np.nan)
# (after fillna later) then, once, before saving:
mx = np.finfo(np.float32).max
df[feat_cols] = df[feat_cols].clip(lower=-mx, upper=mx)

print('Single-feature discriminative power (AUC-like via 1-depth tree accuracy):')
X_all, y_all = df[feat_cols], df['label']
suspicious = []
for col in feat_cols:
    stump = DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE)
    # subsample for speed
    idx = X_all.sample(min(50000, len(X_all)), random_state=RANDOM_STATE).index
    stump.fit(X_all.loc[idx, [col]], y_all.loc[idx])
    acc = stump.score(X_all.loc[idx, [col]], y_all.loc[idx])
    # baseline accuracy of always-majority on this subsample
    base = max(y_all.loc[idx].mean(), 1 - y_all.loc[idx].mean())
    if acc > base + 0.30:   # single feature beats majority baseline by a lot
        suspicious.append((col, round(acc, 3)))

if suspicious:
    print('\n*** Potentially leaky single features (review before trusting results): ***')
    for c, a in sorted(suspicious, key=lambda x: -x[1]):
        print(f'  {c}: single-feature accuracy {a}')
else:
    print('\nNo single feature dominates — good sign.')

Single-feature discriminative power (AUC-like via 1-depth tree accuracy):

No single feature dominates — good sign.


## 4. Log-scale skewed features

Value/gas features span many orders of magnitude. We apply a signed log1p so CTGAN and the classifiers see compressed, well-behaved distributions. (We keep raw columns too is unnecessary — we transform in place.)

In [ ]:
feat_cols = [c for c in df.columns if c != 'label']
skew = df[feat_cols].skew().abs().sort_values(ascending=False)
skewed = skew[skew > 2].index.tolist()
print('Log-transforming', len(skewed), 'highly-skewed features:')
print(skewed)

def signed_log1p(x):
    return np.sign(x) * np.log1p(np.abs(x))

for col in skewed:
    df[col] = signed_log1p(df[col])

print('\nDone. Final feature count:', len(feat_cols))

Log-transforming 60 highly-skewed features:
['effective_gas_price', 'values.values_minimum_val', 'total_gas_cost', 'value', 'gas_price_ratio', 'length_log', 'gas_used.gas_used_minimum_val', 'nonce.nonce_minimum_val', 'gas_used', 'values.values_median', 'cumulativeGasUsed.cumulativeGasUsed_minimum_val', 'gas_used.gas_used_variance', 'is_same_address', 'gas_prices.gas_prices_variance', 'gas_prices.gas_prices_maximum_val', 'gas_prices.gas_prices_range_value', 'gas_prices.gas_prices_standard_deviation', 'gas_prices.gas_prices_mode', 'length_to', 'nonce.nonce_median', 'gas_prices.gas_prices_summation', 'gas_prices.gas_prices_average', 'values.values_mode', 'gas_used.gas_used_range_value', 'gas_used.gas_used_maximum_val', 'gas_used.gas_used_median', 'cumulativeGasUsed.cumulativeGasUsed_skewness', 'gas_prices.gas_prices_median', 'values.values_average', 'gas_used.gas_used_summation', 'gas_prices.gas_prices_coefficient_of_variation', 'values.values_standard_deviation', 'values.values_maximum_v

## 5. Train / Validation / Test split — real fraud held out for test

**The critical safeguard.** We split BEFORE any down-sampling or augmentation, stratified by label. The test set keeps REAL fraud and is never touched by synthetic data. Standardization is fit on TRAIN only.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

feat_cols = [c for c in df.columns if c != 'label']
X, y = df[feat_cols], df['label']

# 70 / 15 / 15 stratified
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.1765, stratify=y_tmp, random_state=RANDOM_STATE)  # 0.1765*0.85 ~= 0.15

print('Train:', X_train.shape, 'fraud:', int(y_train.sum()))
print('Val:  ', X_val.shape,   'fraud:', int(y_val.sum()))
print('Test: ', X_test.shape,  'fraud:', int(y_test.sum()), '<- REAL fraud, held out')

# Standardize on train only
scaler = StandardScaler().fit(X_train)
X_train_s = pd.DataFrame(scaler.transform(X_train), columns=feat_cols, index=X_train.index)
X_val_s   = pd.DataFrame(scaler.transform(X_val),   columns=feat_cols, index=X_val.index)
X_test_s  = pd.DataFrame(scaler.transform(X_test),  columns=feat_cols, index=X_test.index)
joblib.dump(scaler, os.path.join(OUT_DIR, 'scaler.joblib'))

Train: (718780, 74) fraud: 62153
Val:   (154056, 74) fraud: 13321
Test:  (154031, 74) fraud: 13319 <- REAL fraud, held out


['/content/drive/MyDrive/SDG_paper/data/processed/scaler.joblib']

## 6. Construct the imbalance sweep (1:100, 1:500, 1:1000)

For each target ratio we down-sample the **training** fraud so that fraud:legit = 1:ratio. Validation and test are left untouched (real distribution). Each training set is saved separately; notebook 3 trains a generator on each.

> Note: at 1:1000 the available training fraud may be the limiting factor; the cell reports the realized ratio.

In [ ]:
def make_imbalanced_training(X_tr, y_tr, ratio, seed=RANDOM_STATE):
    """Down-sample the fraud (minority) class so fraud:legit = 1:ratio."""
    rng = np.random.RandomState(seed)
    legit_idx = y_tr[y_tr == 0].index
    fraud_idx = y_tr[y_tr == 1].index
    n_legit = len(legit_idx)
    target_fraud = max(1, int(round(n_legit / ratio)))
    if target_fraud >= len(fraud_idx):
        kept_fraud = fraud_idx  # not enough fraud to reach the ratio; keep all
    else:
        kept_fraud = rng.choice(fraud_idx, size=target_fraud, replace=False)
    keep = np.concatenate([legit_idx.values, np.asarray(kept_fraud)])
    rng.shuffle(keep)
    realized = n_legit / len(kept_fraud)
    return X_tr.loc[keep], y_tr.loc[keep], realized

ratios = [100, 500, 1000]
for r in ratios:
    Xr, yr, realized = make_imbalanced_training(X_train_s, y_train, r)
    tag = f'1_{r}'
    Xr.assign(label=yr).to_parquet(os.path.join(OUT_DIR, f'train_{tag}.parquet'))
    print(f'ratio 1:{r:<4} -> {len(Xr):>7} rows, fraud={int(yr.sum()):>4}, realized 1:{realized:.0f}')

# Save val and test once (shared across ratios)
X_val_s.assign(label=y_val).to_parquet(os.path.join(OUT_DIR, 'val.parquet'))
X_test_s.assign(label=y_test).to_parquet(os.path.join(OUT_DIR, 'test.parquet'))

# Save the feature list for downstream notebooks
with open(os.path.join(OUT_DIR, 'feature_cols.json'), 'w') as f:
    json.dump(feat_cols, f)

print('\nSaved train_1_100 / train_1_500 / train_1_1000, val, test, scaler, feature_cols to', OUT_DIR)

ratio 1:100  ->  663193 rows, fraud=6566, realized 1:100
ratio 1:500  ->  657940 rows, fraud=1313, realized 1:500
ratio 1:1000 ->  657284 rows, fraud= 657, realized 1:999

Saved train_1_100 / train_1_500 / train_1_1000, val, test, scaler, feature_cols to /content/drive/MyDrive/SDG_paper/data/processed
